# Data Analysis Agent

> Ask questions about your data in plain English. The agent writes the code, runs it, draws the charts, and explains the results.

## The Problem

Most people who own data cannot query it. A shop owner has a sales CSV, a clinic has patient visit records, a farmer's cooperative has yield spreadsheets, but answering a simple question like *"Which products sold worst last quarter?"* still means finding someone who knows pandas or SQL.

This project builds an **AI agent** that closes that gap.

## What It Does

You load a dataset and ask a question. The agent then works in a loop:

1. **Reads** a summary of your data (columns, types, sample rows)
2. **Writes** Python (pandas, matplotlib) to answer the question
3. **Runs** the code and looks at the output
4. **Fixes** its own mistakes if the code errors
5. **Explains** the result in plain English, based only on what the code actually printed

Example questions:

- "Which 5 products sold worst last quarter?"
- "Plot monthly revenue and tell me if there's a trend."
- "Are there missing values or outliers I should worry about?"
- "Now break that down by region." *(follow-ups keep context)*

## Why an Agent, Not Just a Chatbot?

A chatbot guesses at answers from its training data. This agent **does the work**: it computes on your real data, so every number in its answer is traceable to code you can read. If its first attempt fails (a wrong column name, a bad date format), it reads the error and tries again, which is what makes it an agent rather than a one-shot text generator.

## How This Notebook Is Organized

This is not a flat pile of cells. The notebook is structured as a **production-grade project in seven layers**, each with one job. Some layers are code you run; others use `%%writefile` cells to generate the real project files on disk, so running the notebook top to bottom produces a proper repository.

| # | Layer | Job | What it produces |
|---|-------|-----|------------------|
| 1 | **Foundation** | Project identity, dependencies, docs | Folder structure, `requirements.txt`, `README.md`, `LICENSE` |
| 2 | **Configuration** | Environment-specific secrets and settings | `.env.example`, `.gitignore`, `config.py` |
| 3 | **Source** | The actual agent logic | `tools.py`, `prompts.py`, `agent.py`, `main.py` |
| 4 | **Containerization** | Ship and run anywhere | `Dockerfile`, `docker-compose.yml` |
| 5 | **Observability** | See what it is doing | Structured JSON logs of every question, step, error, and token count |
| 6 | **Evaluation** | Prove it works, before and after every change | Test questions with known answers, unit tests |
| 7 | **Tooling & Automation** | CI/CD, editor config, scripts | GitHub Actions workflow, `Makefile`, `.editorconfig` |

## Tech Stack

- **Language:** Python 3.12
- **Model:** Llama 3.3 70B via the [Groq](https://console.groq.com/keys) API (free tier available, supports tool calling)
- **Data:** pandas, matplotlib (DuckDB planned for SQL)
- **Config:** python-dotenv, pydantic-settings
- **Quality:** pytest, GitHub Actions
- **Packaging:** Docker

The model is swappable. Changing the client and model name in Layer 2 and Layer 3 is enough to move to Claude, OpenAI, or a local model.

## Before You Start

1. Create a free API key at **console.groq.com/keys**
2. Have a CSV ready to analyze (any tabular data works)
3. Python 3.10+ and Jupyter installed

## A Note on Safety

The agent executes code that the model writes. In a personal notebook with data you trust, that is fine, but **`exec()` is not a sandbox**: the code runs with the same access as your Python session. Layer 4 (Docker) is what contains it. Do not expose this agent to other users without running it inside a container or a dedicated sandbox.

## Roadmap

- [ ] Add a SQL tool (DuckDB) so the agent can choose between pandas and SQL
- [ ] File upload widget for non-technical users
- [ ] A data-cleaning tool that describes its changes before applying them
- [ ] Larger evaluation set with real expected answers
- [ ] Hosted observability (Langfuse or LangSmith)

## Author

*Willard Odongo*

## Layer 1: Foundation

This is the project’s identity — the stuff that answers “what is this, who owns it, how do I install it.”

In [1]:
from pathlib import Path
for d in ["src/agent", "tests", "evals", "data", "logs", ".github/workflows"]:
    Path(d).mkdir(parents=True, exist_ok=True)
Path("src/agent/__init__.py").touch()

In [2]:
%%writefile requirements.txt
pandas
matplotlib
plotly
kaleido
pillow
groq
streamlit>=1.50
openpyxl
fpdf2
tableauhyperapi; sys_platform == "linux" and platform_machine == "x86_64"
python-dotenv
pydantic-settings
duckdb
pytest


Overwriting requirements.txt


In [3]:
%%writefile LICENSE
MIT License. Willard Odongo, 2026.

Overwriting LICENSE


## Layer 2: Configuration

AI agents talk to APIs — model providers, vector stores, external tools. Each environment needs its own keys and settings, and none of them belong in source code.

In [4]:
%%writefile .env.example
GROQ_API_KEY=your_api_key_here
MODEL=openai/gpt-oss-120b
MAX_STEPS=6
LOG_LEVEL=INFO

Overwriting .env.example


In [5]:
%%writefile .gitignore
.env
.venv/
logs/
data/
__pycache__/
.ipynb_checkpoints/

Overwriting .gitignore


In [6]:
%%writefile src/agent/config.py
from pydantic_settings import BaseSettings

class Settings(BaseSettings):
    groq_api_key: str
    model: str = "openai/gpt-oss-120b"
    max_steps: int = 6
    log_level: str = "INFO"

    class Config:
        env_file = ".env"

settings = Settings()

Overwriting src/agent/config.py


## Layer 3:Source

`src/` — the actual agent: graph/workflow definitions, tools, prompts, retrieval logic, API layer. Everything that makes the agent an agent lives here, cleanly separated from infra and tooling.

In [7]:
%%writefile src/agent/tools.py
import io, os, contextlib, traceback
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

FIGURES = []   # every chart the agent makes lands here

def show_chart(fig):
    os.makedirs("logs", exist_ok=True)
    FIGURES.append(fig)
    path = "logs/last_chart.html"
    fig.write_html(path)          # saved for CLI / Docker / notebook use
    print(f"(chart saved to {path})")

show = show_chart                 # alias, in case the model calls show(fig)

def run_python(code: str, df: pd.DataFrame) -> str:
    out = io.StringIO()
    env = {"df": df, "pd": pd, "px": px, "go": go, "show_chart": show_chart, "show": show}
    try:
        with contextlib.redirect_stdout(out):
            exec(code, env)
    except Exception:
        return out.getvalue() + "\nERROR:\n" + traceback.format_exc(limit=2)
    return out.getvalue() or "(ran fine, nothing printed, use print())"

TOOLS = [{
    "type": "function",
    "function": {
        "name": "run_python",
        "description": "Run Python against the DataFrame `df`. Returns printed output or an error.",
        "parameters": {
            "type": "object",
            "properties": {"code": {"type": "string"}},
            "required": ["code"],
        },
    },
}]

Overwriting src/agent/tools.py


In [8]:
%%writefile src/agent/prompts.py
import io

def describe_df(df):
    buf = io.StringIO()
    df.info(buf=buf)
    return f"{buf.getvalue()}\nSample rows:\n{df.head(3).to_string()}"

def build_system_prompt(df):
    return f"""You are a data analyst. A pandas DataFrame named `df` is loaded.
Answer by writing Python and calling run_python.
- Use print() to show results; you only see printed output.
- Charts: build with plotly express (`px`) and call `show_chart(fig)`.
  Do not call `fig.show()` - it does nothing in this environment.
- If code errors, read the error and fix it.
- Never invent numbers. Base answers only on tool output.
- End with a short plain-English explanation.

DataFrame info:
{describe_df(df)}"""

Overwriting src/agent/prompts.py


In [9]:
%%writefile src/agent/agent.py
import json, time
from groq import Groq
from .config import settings
from .tools import TOOLS, run_python
from .prompts import build_system_prompt
from .observability import get_logger

log = get_logger()
client = Groq(api_key=settings.groq_api_key)

def ask(question, df, messages=None):
    messages = messages or [{"role": "system", "content": build_system_prompt(df)}]
    messages.append({"role": "user", "content": question})
    trace = {"question": question, "steps": []}

    for step in range(settings.max_steps):
        t0 = time.time()
        resp = client.chat.completions.create(
            model=settings.model, messages=messages, tools=TOOLS, tool_choice="auto"
        )
        msg = resp.choices[0].message
        latency = round(time.time() - t0, 2)

        if not msg.tool_calls:
            trace.update(answer=msg.content, tokens=resp.usage.total_tokens)
            log.info(json.dumps(trace))
            return msg.content, messages

        messages.append({
            "role": "assistant", "content": msg.content or "",
            "tool_calls": [{"id": tc.id, "type": "function",
                            "function": {"name": tc.function.name,
                                         "arguments": tc.function.arguments}}
                           for tc in msg.tool_calls],
        })
        for tc in msg.tool_calls:
            code = json.loads(tc.function.arguments)["code"]
            result = run_python(code, df)
            trace["steps"].append({"code": code, "result": result[:500],
                                   "latency_s": latency, "error": "ERROR" in result})
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": result[:4000]})

    trace["answer"] = None
    log.warning(json.dumps(trace))
    return "Stopped: hit step limit.", messages

Overwriting src/agent/agent.py


In [10]:
%%writefile src/agent/main.py
import sys
import pandas as pd
from .agent import ask

USAGE = 'usage: python -m agent.main <csv> "<question>"'

if __name__ == "__main__":
    if len(sys.argv) < 3 or not sys.argv[2].strip():
        sys.exit(USAGE)
    df = pd.read_csv(sys.argv[1])
    answer, _ = ask(sys.argv[2], df)
    print(answer)

Overwriting src/agent/main.py


## Layer 4: Containerization

`docker/` — Dockerfiles, docker-compose configs, entrypoint scripts. This is what makes "works on my machine" irrelevant — the agent runs identically in dev, staging, and prod.

In [11]:
%%writefile Dockerfile
FROM python:3.12-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
# Chromium lets kaleido render chart PNGs (used by the Streamlit export button).
RUN apt-get update \
    && apt-get install -y --no-install-recommends chromium \
    && rm -rf /var/lib/apt/lists/*
COPY src/ src/
COPY dashboard/ dashboard/
COPY app.py .
ENV PYTHONPATH=/app/src
ENTRYPOINT ["python", "-m", "agent.main"]


Overwriting Dockerfile


In [12]:
%%writefile docker-compose.yml
services:
  agent:
    build: .
    env_file: .env
    volumes:
      - ./data:/app/data:ro
      - ./logs:/app/logs

  ui:
    build: .
    entrypoint: ["streamlit", "run", "app.py", "--server.address=0.0.0.0", "--server.port=8501"]
    env_file: .env
    ports:
      - "8501:8501"
    volumes:
      - ./data:/app/data:ro
      - ./logs:/app/logs

Overwriting docker-compose.yml


## Layer 5: Observability

If you can’t see what your agent is doing in production, you don’t have a production system — you have a black box you’re hoping doesn’t break.

In [13]:
%%writefile src/agent/observability.py
import logging, os

def get_logger():
    os.makedirs("logs", exist_ok=True)
    logger = logging.getLogger("agent")
    if not logger.handlers:
        logger.setLevel(os.getenv("LOG_LEVEL", "INFO"))
        logger.addHandler(logging.FileHandler("logs/agent.jsonl"))
        logger.addHandler(logging.StreamHandler())
    return logger

Overwriting src/agent/observability.py


In [14]:
from pathlib import Path
import pandas as pd

log_path = Path("logs/agent.jsonl")
if not log_path.exists() or log_path.stat().st_size == 0:
    print("No agent runs logged yet - run a question first.")
else:
    pd.read_json(log_path, lines=True).tail()

## Layer 6: Evaluation

`evals/` — this is what separates a serious AI project from a toy. Automated evaluation sets, golden datasets, scoring scripts, regression checks that run whenever the prompt, model, or logic changes. Without this layer, every change to the agent is a guess.

In [15]:
%%writefile evals/cases.json
[
  {"question": "How many rows are in the data?", "expected": "100"},
  {"question": "What is the total revenue?", "expected": "46269.26"}
]

Overwriting evals/cases.json


In [16]:
%%writefile evals/run_evals.py
import json, sys
import pandas as pd
sys.path.insert(0, "src")
from agent.agent import ask

df = pd.read_csv(sys.argv[1])
cases = json.load(open("evals/cases.json"))
passed = 0
for c in cases:
    answer, _ = ask(c["question"], df)
    ok = c["expected"] in (answer or "").replace(",", "")
    passed += ok
    print(("PASS" if ok else "FAIL"), "-", c["question"])
print(f"\n{passed}/{len(cases)} passed")

Overwriting evals/run_evals.py


In [17]:
%%writefile tests/test_tools.py
import sys; sys.path.insert(0, "src")
import pandas as pd
from agent.tools import run_python

def test_runs_code():
    df = pd.DataFrame({"a": [1, 2, 3]})
    assert "6" in run_python("print(df.a.sum())", df)

def test_reports_errors():
    df = pd.DataFrame({"a": [1]})
    assert "ERROR" in run_python("print(df.missing)", df)

Overwriting tests/test_tools.py


## Layer 7: Tooling & Automation

`.github/` — CI/CD workflows: linting, tests, evals, and deployment pipelines running automatically on every push/PR

In [18]:
%%writefile .github/workflows/ci.yml
name: CI

on: [push, pull_request]

concurrency:
  group: ci-${{ github.ref }}
  cancel-in-progress: true

jobs:
  test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.12"
          cache: pip
      - run: pip install -r requirements.txt
      - run: python -m py_compile app.py src/agent/*.py dashboard/*.py
      - run: pytest tests/ -q

  docker:
    runs-on: ubuntu-latest
    needs: test
    steps:
      - uses: actions/checkout@v4
      - uses: docker/setup-buildx-action@v3
      - name: Build image (no push)
        uses: docker/build-push-action@v6
        with:
          context: .
          push: false
          load: true
          tags: data-ai-agent:ci
          cache-from: type=gha
          cache-to: type=gha,mode=max

  publish:
    # CD: publish to GHCR only when you push a version tag (git tag v1.0.0 && git push --tags)
    if: startsWith(github.ref, 'refs/tags/v')
    runs-on: ubuntu-latest
    needs: [test, docker]
    permissions:
      contents: read
      packages: write
    steps:
      - uses: actions/checkout@v4
      - name: Lowercase image name
        run: echo "IMAGE=ghcr.io/${GITHUB_REPOSITORY,,}" >> "$GITHUB_ENV"
      - uses: docker/setup-buildx-action@v3
      - uses: docker/login-action@v3
        with:
          registry: ghcr.io
          username: ${{ github.actor }}
          password: ${{ secrets.GITHUB_TOKEN }}
      - uses: docker/metadata-action@v5
        id: meta
        with:
          images: ${{ env.IMAGE }}
          tags: |
            type=semver,pattern={{version}}
            type=sha
      - uses: docker/build-push-action@v6
        with:
          context: .
          push: true
          tags: ${{ steps.meta.outputs.tags }}
          labels: ${{ steps.meta.outputs.labels }}
          cache-from: type=gha
          cache-to: type=gha,mode=max


Overwriting .github/workflows/ci.yml


In [19]:
%%writefile Makefile
PY ?= .venv/bin/python
export PYTHONPATH := src

.PHONY: install test eval run ui docker docker-ui

install:
	$(PY) -m pip install -r requirements.txt

test:
	$(PY) -m pytest tests/ -q

eval:
	$(PY) evals/run_evals.py data/sales.csv

run:
	$(PY) -m agent.main data/sales.csv "$(Q)"

ui:
	$(PY) -m streamlit run app.py

docker:
	docker compose run --rm agent data/sales.csv "$(Q)"

docker-ui:
	docker compose up ui

Overwriting Makefile


In [20]:
%%writefile .editorconfig
root = true
[*]
indent_style = space
indent_size = 4
end_of_line = lf
insert_final_newline = true

Overwriting .editorconfig


## Layer 8: Dashboard engine

Four small modules behind the Dashboard tab. The model never writes plotting code here: it returns a JSON dashboard spec, the app validates it, and fixed templates render it.

- `dashboard/profile.py` - column kinds, cardinality, missing rates, ranges and the date span; only this profile plus three sample rows go to the model
- `dashboard/planner.py` - prompt, JSON parsing, spec validation (bad charts are dropped), and a rule-based fallback dashboard
- `dashboard/render.py` - spec to Plotly figures, KPI cards and one-line insights; sidebar filters apply locally and never call the model
- `dashboard/export.py` - PNG, PDF, HTML, Excel, CSV zip, Tableau Hyper and JSON, with honest availability checks


In [ ]:
%%writefile dashboard/__init__.py
"""Dashboard package: profile, plan, render, export."""
from . import export, planner, profile, render

__all__ = ["export", "planner", "profile", "render"]


In [ ]:
%%writefile dashboard/profile.py
"""Data profiling for the dashboard planner.

Turns a DataFrame into a compact, JSON-safe description the planner can read:
column kinds, cardinality, missing rates, numeric ranges and the date span.
Only this profile (plus a few sample rows) is ever sent to the model.
"""
from __future__ import annotations

import math
import re

import pandas as pd

SAMPLE_ROWS = 3
MAX_CATEGORICAL_CARDINALITY = 50
MAX_MISSING_PCT = 50.0

_ID_PATTERN = re.compile(r"(^id$|_id$|id$|^index$|^uuid$|_no$|_code$)", re.IGNORECASE)
_DATE_HINTS = ("date", "time", "month", "year", "day", "week")


def pretty_label(name: str) -> str:
    """`unit_price` -> `Unit Price`."""
    text = re.sub(r"[_\-]+", " ", str(name)).strip()
    return " ".join(word.capitalize() if word.islower() else word for word in text.split())


def _json_number(value):
    if value is None:
        return None
    try:
        number = float(value)
    except (TypeError, ValueError):
        return None
    if math.isnan(number) or math.isinf(number):
        return None
    return round(number, 4)


def _name_is_id(name: str) -> bool:
    return bool(_ID_PATTERN.search(str(name)))


def looks_like_date(series: pd.Series) -> bool:
    if pd.api.types.is_datetime64_any_dtype(series):
        return True
    if pd.api.types.is_numeric_dtype(series):
        return False
    name = str(series.name).lower()
    if not any(hint in name for hint in _DATE_HINTS):
        return False
    sample = series.dropna().head(20)
    if sample.empty:
        return False
    parsed = pd.to_datetime(sample, errors="coerce")
    return bool(parsed.notna().mean() >= 0.8)


def column_kind(df: pd.DataFrame, col: str) -> str:
    series = df[col]
    if looks_like_date(series):
        return "date"
    if pd.api.types.is_bool_dtype(series):
        return "boolean"
    if pd.api.types.is_numeric_dtype(series):
        return "numeric"
    if series.nunique(dropna=True) <= MAX_CATEGORICAL_CARDINALITY:
        return "categorical"
    return "text"


def profile_dataframe(df: pd.DataFrame) -> dict:
    """Return a JSON-safe profile of ``df``."""
    rows = len(df)
    columns: list[dict] = []
    date_range: dict | None = None

    for col in df.columns:
        series = df[col]
        kind = column_kind(df, col)
        nunique = int(series.nunique(dropna=True))
        missing_pct = round(float(series.isna().mean()) * 100, 1) if rows else 0.0

        info: dict = {
            "name": str(col),
            "kind": kind,
            "dtype": str(series.dtype),
            "missing_pct": missing_pct,
            "nunique": nunique,
            # Uniqueness only signals an ID for text-like columns: a numeric
            # measure can legitimately be unique per row (e.g. revenue).
            "is_id": bool(
                _name_is_id(col)
                or (kind in ("categorical", "text") and rows > 20 and nunique == rows)
            ),
            "usable": missing_pct <= MAX_MISSING_PCT,
        }

        if kind == "numeric":
            info["min"] = _json_number(series.min())
            info["max"] = _json_number(series.max())
            info["mean"] = _json_number(series.mean())

        if kind in ("categorical", "boolean"):
            info["top_values"] = [str(v) for v in series.dropna().value_counts().head(5).index]

        if kind == "date":
            parsed = pd.to_datetime(series, errors="coerce")
            if parsed.notna().any():
                info["min"] = str(parsed.min().date())
                info["max"] = str(parsed.max().date())
                if date_range is None:
                    date_range = {
                        "column": str(col),
                        "min": info["min"],
                        "max": info["max"],
                    }

        columns.append(info)

    sample_rows = [
        {str(key): str(value) for key, value in row.items()}
        for row in df.head(SAMPLE_ROWS).to_dict(orient="records")
    ]

    return {
        "rows": rows,
        "column_count": len(df.columns),
        "date_range": date_range,
        "columns": columns,
        "sample_rows": sample_rows,
    }


def column_map(profile: dict) -> dict[str, dict]:
    return {col["name"]: col for col in profile["columns"]}


def usable_columns(profile: dict, *kinds: str) -> list[str]:
    wanted = set(kinds) if kinds else None
    return [
        col["name"]
        for col in profile["columns"]
        if col["usable"] and not col["is_id"] and (wanted is None or col["kind"] in wanted)
    ]


def describe_profile(profile: dict) -> str:
    """One-sentence summary used for subtitles and fallback specs."""
    rows = profile["rows"]
    parts = [f"{rows:,} rows", f"{profile['column_count']} columns"]
    span = profile.get("date_range")
    if span:
        parts.append(f"{span['min']} to {span['max']}")
    return " · ".join(parts)


In [ ]:
%%writefile dashboard/planner.py
"""Dashboard planning.

The planner asks the model for a JSON dashboard spec (never plotting code),
validates it against the data profile, and falls back to a rule-based
dashboard when the model returns something unusable.
"""
from __future__ import annotations

import json
import time

from .profile import (
    column_map,
    describe_profile,
    pretty_label,
    usable_columns,
)

ALLOWED_TYPES = (
    "line",
    "bar",
    "hbar",
    "stacked_bar",
    "pie",
    "scatter",
    "histogram",
    "box",
    "heatmap",
    "table",
)
ALLOWED_AGGS = ("sum", "mean", "count", "median")
ALLOWED_FORMATS = ("currency", "number", "percent")
MAX_CHARTS = 6
MAX_KPIS = 4
MAX_FILTERS = 3
MAX_TOP_N = 50
DEFAULT_TOP_N = 10
MAX_PIE_SLICES = 5

MONEY_HINTS = ("revenue", "sales", "amount", "total", "price", "value", "cost", "profit")

SYSTEM_PROMPT = """You are a dashboard planner for a business user who does not code.
You receive a data profile and return ONLY a JSON object describing a dashboard.
You never write Python or plotting code.

Schema:
{
  "title": "string",
  "subtitle": "one sentence describing the data and date range",
  "kpis": [{"label": "Total revenue", "column": "revenue", "agg": "sum|mean|count|median", "format": "currency|number|percent"}],
  "charts": [{"type": "line|bar|hbar|stacked_bar|pie|scatter|histogram|box|heatmap|table",
              "title": "string", "x": "column", "y": "column", "color": "column or null",
              "agg": "sum|mean|count|median", "top_n": 10,
              "insight": "one plain-English sentence supported by the data"}],
  "filters": ["up to 3 categorical or date columns"]
}

Rules:
- Use ONLY columns present in the profile. Never invent names.
- Skip ID-like columns and columns with more than 50% missing values.
- Date column + numeric measure -> line chart, aggregated by month (or week for short spans).
- Category + numeric measure -> horizontal bar, sorted, top 10 plus an "Other" bucket.
- Parts of a whole with 5 or fewer categories -> pie; more than 5 -> bar.
- Two categories + one measure -> stacked_bar or heatmap.
- Two numeric columns -> scatter, colored by a category when one exists.
- Single numeric column -> histogram; comparing groups -> box.
- Never use 3D charts, dual axes, or more than 6 charts.
- 3 to 4 KPIs. Prefer fewer, clearer visuals.
- Every insight must be checkable from the data. Do not invent numbers.
Return JSON only, with no markdown fences or commentary."""


def build_prompt(profile: dict, focus: str | None = None) -> str:
    payload = {
        "profile": profile,
        "focus": (focus or "").strip() or None,
    }
    return (
        "Build a dashboard spec from this profile.\n\n"
        + json.dumps(payload, indent=2)
        + "\n\nReturn only the JSON object."
    )


def parse_json(text: str) -> dict:
    """Parse JSON, tolerating markdown fences or leading prose."""
    if not text:
        raise ValueError("empty response")
    cleaned = text.strip()
    if cleaned.startswith("```"):
        cleaned = cleaned.split("\n", 1)[1] if "\n" in cleaned else ""
        if cleaned.rstrip().endswith("```"):
            cleaned = cleaned.rstrip()[:-3]
    start, end = cleaned.find("{"), cleaned.rfind("}")
    if start == -1 or end == -1:
        raise ValueError("no JSON object found")
    return json.loads(cleaned[start : end + 1])


def _text(value, default: str, limit: int = 200) -> str:
    if not isinstance(value, str) or not value.strip():
        return default
    return value.strip()[:limit]


def _is_money(column: str) -> bool:
    name = str(column).lower()
    return any(hint in name for hint in MONEY_HINTS)


TOTAL_HINTS = ("revenue", "sales", "amount", "total", "profit", "spend", "income", "cost")
RATE_HINTS = ("price", "rate", "pct", "percent", "ratio", "unit")


def _measure_rank(column: str) -> int:
    """Rank measures so totals lead: revenue > generic numeric > unit price."""
    name = str(column).lower()
    if any(hint in name for hint in TOTAL_HINTS):
        return 0
    if any(hint in name for hint in RATE_HINTS):
        return 2
    return 1


def _primary_measure(numeric: list[str]) -> str | None:
    if not numeric:
        return None
    return min(numeric, key=lambda col: (_measure_rank(col), numeric.index(col)))


def _kpi_agg_and_format(column: str) -> tuple[str, str]:
    name = str(column).lower()
    if _is_money(column):
        if any(key in name for key in ("price", "rate", "unit")):
            return "mean", "currency"
        return "sum", "currency"
    if any(key in name for key in ("pct", "percent", "rate", "ratio")):
        return "mean", "percent"
    return "sum", "number"


def fallback_kpis(profile: dict) -> list[dict]:
    numeric = usable_columns(profile, "numeric")
    ordered = sorted(numeric, key=lambda col: (_measure_rank(col), numeric.index(col)))
    kpis = []
    for column in ordered[:MAX_KPIS]:
        agg, fmt = _kpi_agg_and_format(column)
        kpis.append(
            {
                "label": pretty_label(column),
                "column": column,
                "agg": agg,
                "format": fmt,
            }
        )
    if len(kpis) < 3 and profile["columns"]:
        kpis.append(
            {
                "label": "Rows",
                "column": profile["columns"][0]["name"],
                "agg": "count",
                "format": "number",
            }
        )
    return kpis[:MAX_KPIS]


def _chart(ctype: str, title: str, x, y, color=None, agg="sum", top_n=DEFAULT_TOP_N, insight=""):
    return {
        "type": ctype,
        "title": title,
        "x": x,
        "y": y,
        "color": color,
        "agg": agg,
        "top_n": top_n,
        "insight": insight,
    }


def fallback_spec(profile: dict) -> dict:
    """Rule-based dashboard used when the model output cannot be used."""
    cols = column_map(profile)
    numeric = usable_columns(profile, "numeric")
    categorical = [
        c
        for c in usable_columns(profile, "categorical", "boolean")
        if 1 < cols[c]["nunique"] <= 50
    ]
    dates = usable_columns(profile, "date")
    date_col = dates[0] if dates else None
    primary = _primary_measure(numeric)

    charts: list[dict] = []
    if date_col and primary:
        charts.append(
            _chart("line", f"{pretty_label(primary)} over time", date_col, primary)
        )
    if categorical and primary:
        charts.append(
            _chart(
                "hbar",
                f"{pretty_label(primary)} by {pretty_label(categorical[0])}",
                categorical[0],
                primary,
                top_n=DEFAULT_TOP_N,
            )
        )
    if len(categorical) >= 2 and primary:
        charts.append(
            _chart(
                "stacked_bar",
                f"{pretty_label(primary)} by {pretty_label(categorical[0])} and {pretty_label(categorical[1])}",
                categorical[0],
                primary,
                color=categorical[1],
            )
        )
    others = [c for c in numeric if c != primary]
    secondary = min(others, key=lambda col: (_measure_rank(col), numeric.index(col))) if others else None

    if primary and secondary:
        charts.append(
            _chart(
                "scatter",
                f"{pretty_label(primary)} vs {pretty_label(secondary)}",
                secondary,
                primary,
                color=categorical[0] if categorical else None,
            )
        )
    if secondary:
        charts.append(
            _chart("histogram", f"Distribution of {pretty_label(secondary)}", secondary, None)
        )
    if categorical and primary:
        charts.append(
            _chart(
                "box",
                f"{pretty_label(primary)} by {pretty_label(categorical[0])}",
                categorical[0],
                primary,
            )
        )
    if categorical and len(charts) < MAX_CHARTS:
        last = categorical[-1]
        ctype = "pie" if cols[last]["nunique"] <= MAX_PIE_SLICES else "bar"
        charts.append(
            _chart(ctype, f"Share by {pretty_label(last)}", last, primary)
        )

    # De-duplicate identical (type, x, y) triples.
    seen, unique = set(), []
    for chart in charts:
        key = (chart["type"], chart["x"], chart["y"])
        if key in seen:
            continue
        seen.add(key)
        unique.append(chart)

    filters = ([date_col] if date_col else []) + categorical[:2]

    return {
        "title": "Data overview",
        "subtitle": describe_profile(profile),
        "kpis": fallback_kpis(profile),
        "charts": unique[:MAX_CHARTS],
        "filters": filters[:MAX_FILTERS],
    }


def _validate_chart(chart, cols: dict) -> tuple[dict | None, str | None]:
    if not isinstance(chart, dict):
        return None, "chart is not an object"

    ctype = chart.get("type")
    if ctype not in ALLOWED_TYPES:
        return None, f"unsupported chart type {ctype!r}"

    x, y, color = chart.get("x"), chart.get("y"), chart.get("color")

    for value, label in ((x, "x"), (y, "y")):
        if value not in (None, "") and value not in cols:
            return None, f"unknown {label} column {value!r}"

    if color not in (None, "") and color not in cols:
        color = None

    def kind(column):
        return cols[column]["kind"] if column in cols else None

    def is_dimension(column):
        return kind(column) in ("categorical", "boolean", "date")

    def is_measure(column):
        return kind(column) == "numeric"

    # Models sometimes put the measure on x for horizontal bars (or swap the
    # axes for line/box). Swap back rather than dropping a usable chart.
    if ctype in ("bar", "hbar", "stacked_bar", "pie", "line", "box"):
        if x and y and is_measure(x) and is_dimension(y):
            x, y = y, x

    if ctype == "line" and not (kind(x) == "date" and kind(y) == "numeric"):
        return None, "line needs a date x and a numeric y"
    if ctype in ("bar", "hbar", "stacked_bar", "pie"):
        if not is_dimension(x) or kind(y) != "numeric":
            return None, f"{ctype} needs a category x and a numeric y"
    if ctype == "scatter" and not (kind(x) == "numeric" and kind(y) == "numeric"):
        return None, "scatter needs two numeric columns"
    if ctype == "histogram" and kind(x) != "numeric":
        return None, "histogram needs a numeric x"
    if ctype == "box" and kind(y) != "numeric":
        return None, "box needs a numeric y"
    if ctype == "heatmap" and not (
        kind(x) in ("categorical", "boolean", "date")
        and kind(y) in ("categorical", "boolean", "date")
    ):
        return None, "heatmap needs two categorical columns"
    if ctype == "table" and x in (None, "") and y in (None, ""):
        return None, "table needs at least one column"

    # Pie charts with too many slices are downgraded rather than dropped.
    if ctype == "pie" and cols[x]["nunique"] > MAX_PIE_SLICES:
        ctype = "bar"

    agg = chart.get("agg") if chart.get("agg") in ALLOWED_AGGS else "sum"
    if agg != "count" and kind(y) != "numeric":
        agg = "count"

    try:
        top_n = int(chart.get("top_n") or DEFAULT_TOP_N)
    except (TypeError, ValueError):
        top_n = DEFAULT_TOP_N
    top_n = max(1, min(MAX_TOP_N, top_n))

    default_title = (
        f"{pretty_label(y)} by {pretty_label(x)}"
        if y
        else pretty_label(x or "Overview")
    )

    return (
        {
            "type": ctype,
            "title": _text(chart.get("title"), default_title),
            "x": x or None,
            "y": y or None,
            "color": color or None,
            "agg": agg,
            "top_n": top_n,
            "insight": _text(chart.get("insight"), "", limit=240),
        },
        None,
    )


def validate_spec(spec, profile: dict) -> tuple[dict | None, list[dict]]:
    """Drop anything that references missing columns or incompatible types."""
    if not isinstance(spec, dict):
        return None, [{"item": "spec", "reason": "not a JSON object"}]

    cols = column_map(profile)
    dropped: list[dict] = []

    clean = {
        "title": _text(spec.get("title"), "Data overview"),
        "subtitle": _text(spec.get("subtitle"), describe_profile(profile)),
        "kpis": [],
        "charts": [],
        "filters": [],
    }

    for kpi in spec.get("kpis") or []:
        if len(clean["kpis"]) >= MAX_KPIS:
            break
        if not isinstance(kpi, dict):
            dropped.append({"item": "kpi", "reason": "not an object"})
            continue
        column = kpi.get("column")
        if column not in cols:
            dropped.append({"item": kpi.get("label") or column, "reason": f"unknown column {column!r}"})
            continue
        agg = kpi.get("agg") if kpi.get("agg") in ALLOWED_AGGS else "sum"
        if agg != "count" and cols[column]["kind"] != "numeric":
            dropped.append({"item": kpi.get("label") or column, "reason": "aggregation needs a numeric column"})
            continue
        fmt = kpi.get("format") if kpi.get("format") in ALLOWED_FORMATS else "number"
        clean["kpis"].append(
            {
                "label": _text(kpi.get("label"), pretty_label(column)),
                "column": column,
                "agg": agg,
                "format": fmt,
            }
        )

    if not clean["kpis"]:
        clean["kpis"] = fallback_kpis(profile)

    for chart in spec.get("charts") or []:
        if len(clean["charts"]) >= MAX_CHARTS:
            dropped.append({"item": "chart", "reason": "over the 6-chart limit"})
            break
        valid, reason = _validate_chart(chart, cols)
        if valid is None:
            dropped.append(
                {
                    "item": (chart.get("title") if isinstance(chart, dict) else None) or "chart",
                    "reason": reason,
                }
            )
            continue
        clean["charts"].append(valid)

    for column in spec.get("filters") or []:
        if len(clean["filters"]) >= MAX_FILTERS:
            break
        if column in cols and cols[column]["kind"] in ("categorical", "boolean", "date"):
            if column not in clean["filters"]:
                clean["filters"].append(column)
        else:
            dropped.append({"item": column, "reason": "filter must be categorical or date"})

    return clean, dropped


def plan_dashboard(profile: dict, focus: str | None = None) -> tuple[dict, dict]:
    """Ask the model for a spec; validate it; fall back when unusable.

    Returns ``(spec, meta)`` where meta carries fallback/dropped/token info.
    """
    from groq import Groq

    from agent.config import settings
    from agent.observability import get_logger

    logger = get_logger()
    client = Groq(api_key=settings.groq_api_key)

    started = time.time()
    tokens = 0
    raw_error = None
    parsed = None

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_prompt(profile, focus)},
    ]

    for _attempt in range(2):
        try:
            response = client.chat.completions.create(
                model=settings.model,
                messages=messages,
                temperature=0.2,
            )
            tokens += response.usage.total_tokens
            parsed = parse_json(response.choices[0].message.content or "")
            break
        except Exception as exc:  # bad JSON or API error; retry once
            raw_error = f"{type(exc).__name__}: {exc}"

    clean, dropped = (validate_spec(parsed, profile) if parsed else (None, []))
    used_fallback = not clean or not clean["charts"]
    if used_fallback:
        clean = fallback_spec(profile)

    meta = {
        "fallback": used_fallback,
        "dropped": dropped,
        "tokens": tokens,
        "latency_s": round(time.time() - started, 2),
        "error": raw_error,
        "focus": (focus or "").strip() or None,
    }

    logger.info(
        json.dumps(
            {
                "kind": "dashboard",
                "spec": clean,
                "dropped": dropped,
                "fallback": used_fallback,
                "latency_s": meta["latency_s"],
                "tokens": tokens,
                "error": raw_error,
                "focus": meta["focus"],
            }
        )
    )
    return clean, meta


In [ ]:
%%writefile dashboard/render.py
"""Render a dashboard spec into Plotly figures and KPI values.

Everything here is local and deterministic: filters and re-renders never call
the model. Charts that cannot be built are reported, not raised.
"""
from __future__ import annotations

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from .profile import looks_like_date, pretty_label

ACCENT = "#2563EB"
PALETTE = [ACCENT, "#0EA5E9", "#14B8A6", "#F59E0B", "#8B5CF6", "#EF4444", "#64748B"]
TEMPLATE = "plotly_white"
MAX_SCATTER_POINTS = 2000
MAX_TABLE_ROWS = 200


# --------------------------------------------------------------------------- #
# Formatting
# --------------------------------------------------------------------------- #
def format_compact(value, fmt: str = "number") -> str:
    """Format 1200 as 1.2K and 3400000 as 3.4M."""
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return "—"

    if fmt == "percent":
        number = float(value)
        if abs(number) <= 1:
            number *= 100
        return f"{number:,.1f}%"

    prefix = "$" if fmt == "currency" else ""
    number = float(value)
    magnitude = abs(number)

    if magnitude >= 1e9:
        body = f"{number / 1e9:,.1f}B"
    elif magnitude >= 1e6:
        body = f"{number / 1e6:,.1f}M"
    elif magnitude >= 1e3:
        body = f"{number / 1e3:,.1f}K"
    elif magnitude >= 100 or float(number).is_integer():
        body = f"{number:,.0f}"
    elif magnitude >= 1:
        body = f"{number:,.2f}".rstrip("0").rstrip(".")
    else:
        body = f"{number:.3g}"
    return f"{prefix}{body}"


def _style(fig, height: int = 380):
    fig.update_layout(
        template=TEMPLATE,
        height=height,
        margin=dict(l=10, r=10, t=50, b=10),
        colorway=PALETTE,
        legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0),
        font=dict(family="system-ui, -apple-system, sans-serif", size=13),
        hoverlabel=dict(bgcolor="white"),
    )
    return fig


# --------------------------------------------------------------------------- #
# Filtering and aggregation
# --------------------------------------------------------------------------- #
def apply_filters(df: pd.DataFrame, filters: dict) -> pd.DataFrame:
    """Apply sidebar filters locally. Date ranges are ``(start, end)`` tuples."""
    out = df
    for column, value in (filters or {}).items():
        if value in (None, [], ()) or column not in out.columns:
            continue
        if isinstance(value, tuple) and len(value) == 2:
            dates = pd.to_datetime(out[column], errors="coerce")
            start, end = pd.Timestamp(value[0]), pd.Timestamp(value[1])
            out = out[(dates >= start) & (dates <= end + pd.Timedelta(days=1))]
        elif isinstance(value, (list, tuple, set)):
            if not value:
                continue
            out = out[out[column].astype(str).isin([str(v) for v in value])]
        else:
            out = out[out[column] == value]
    return out


def _aggregate(series: pd.Series, agg: str):
    if agg == "count":
        return series.size
    series = series.dropna()
    if series.empty:
        return float("nan")
    if agg == "mean":
        return series.mean()
    if agg == "median":
        return series.median()
    return series.sum()


def _grouped(df: pd.DataFrame, keys, value: str | None, agg: str) -> pd.Series:
    if isinstance(keys, str):
        keys = [keys]
    grouped = df.groupby(keys, dropna=False)
    if agg == "count" or value is None:
        result = grouped.size()
    else:
        result = grouped[value].apply(lambda s: _aggregate(s, agg))
    result.name = "value"
    return result


def _category_table(df: pd.DataFrame, x: str, y: str | None, agg: str, top_n: int) -> pd.DataFrame:
    series = _grouped(df, x, y, agg).sort_values(ascending=False)
    if len(series) > top_n:
        head = series.head(top_n)
        rest = series.iloc[top_n:]
        series = pd.concat([head, pd.Series({"Other": rest.sum()}, name="value")])
    table = series.reset_index()
    table.columns = [x, "value"]
    return table


def _date_table(df: pd.DataFrame, x: str, y: str, agg: str, color: str | None) -> pd.DataFrame:
    data = df[[c for c in (x, y, color) if c]].copy()
    data[x] = pd.to_datetime(data[x], errors="coerce")
    data = data.dropna(subset=[x])
    if data.empty:
        return pd.DataFrame(columns=[x, "value"])

    span_days = (data[x].max() - data[x].min()).days
    period = "W" if span_days <= 120 else "M"
    data["_period"] = data[x].dt.to_period(period).dt.start_time

    keys = ["_period"] + ([color] if color else [])
    series = _grouped(data, keys, y, agg).reset_index()
    series = series.rename(columns={"_period": x})
    return series.sort_values(x)


# --------------------------------------------------------------------------- #
# Insights
# --------------------------------------------------------------------------- #
def _insight_for(chart: dict, table: pd.DataFrame | None, value_fmt: str = "number") -> str:
    if chart.get("insight"):
        return chart["insight"]
    if table is None or table.empty:
        return ""

    ctype = chart["type"]
    try:
        if ctype in ("bar", "hbar", "pie") and "value" in table:
            ranked = table.sort_values("value", ascending=False)
            top = ranked.iloc[0]
            share = ""
            total = table["value"].sum()
            if ctype == "pie" and total:
                share = f" ({top['value'] / total:.0%} of the total)"
            if str(top.iloc[0]) == "Other":
                return (
                    f"The combined remaining categories total "
                    f"{format_compact(top['value'], value_fmt)}."
                )
            return (
                f"{top.iloc[0]} leads with {format_compact(top['value'], value_fmt)}{share}."
            )
        if ctype == "line" and "value" in table:
            first, last = table.iloc[0], table.iloc[-1]
            if first["value"]:
                change = (last["value"] - first["value"]) / abs(first["value"]) * 100
                direction = "rose" if change >= 0 else "fell"
                return (
                    f"{format_compact(last['value'], value_fmt)} at the latest point, "
                    f"{direction} {abs(change):.0f}% from the start."
                )
            return f"Latest value: {format_compact(last['value'], value_fmt)}."
        if ctype == "scatter" and len(table) > 2:
            x, y = chart.get("x"), chart.get("y")
            if x in table and y in table:
                corr = table[x].corr(table[y])
                if pd.notna(corr):
                    strength = "strong" if abs(corr) >= 0.6 else "moderate" if abs(corr) >= 0.3 else "weak"
                    return f"{strength.capitalize()} {('positive' if corr > 0 else 'negative')} relationship (r = {corr:.2f})."
        if ctype == "histogram" and chart.get("x") in table:
            series = table[chart["x"]].dropna()
            if not series.empty:
                return (
                    f"Median {format_compact(series.median(), value_fmt)}, "
                    f"mean {format_compact(series.mean(), value_fmt)}."
                )
        if ctype == "stacked_bar" and "value" in table:
            top = table.sort_values("value", ascending=False).iloc[0]
            label = " / ".join(
                str(top[column])
                for column in (chart.get("x"), chart.get("color"))
                if column in table
            )
            return f"{label} is the largest segment at {format_compact(top['value'], value_fmt)}."
        if ctype == "box":
            group = chart.get("color") or chart.get("x")
            measure = chart.get("y")
            if group in table and measure in table:
                medians = (
                    table[[group, measure]]
                    .dropna()
                    .groupby(group)[measure]
                    .median()
                    .sort_values(ascending=False)
                )
                if not medians.empty:
                    return (
                        f"{medians.index[0]} has the highest median "
                        f"{pretty_label(measure)} at {format_compact(medians.iloc[0], value_fmt)}."
                    )
    except Exception:
        return ""
    return ""


# --------------------------------------------------------------------------- #
# KPI cards
# --------------------------------------------------------------------------- #
def _kpi_value(df: pd.DataFrame, column: str, agg: str):
    if agg == "count":
        return float(len(df))
    if column not in df.columns:
        return None
    value = _aggregate(df[column], agg)
    return None if pd.isna(value) else float(value)


def _period_delta(df: pd.DataFrame, column: str, agg: str, date_col: str | None):
    if not date_col or date_col not in df.columns:
        return None
    dates = pd.to_datetime(df[date_col], errors="coerce")
    if dates.notna().sum() < 4:
        return None
    cutoff = dates.min() + (dates.max() - dates.min()) / 2
    current = _kpi_value(df[dates > cutoff], column, agg)
    previous = _kpi_value(df[dates <= cutoff], column, agg)
    if not current or not previous:
        return None
    return (current - previous) / abs(previous) * 100


def kpi_cards(spec: dict, df: pd.DataFrame, date_col: str | None = None) -> list[dict]:
    cards = []
    for kpi in spec.get("kpis", []):
        value = _kpi_value(df, kpi["column"], kpi.get("agg", "sum"))
        cards.append(
            {
                "label": kpi.get("label", pretty_label(kpi["column"])),
                "value": format_compact(value, kpi.get("format", "number")),
                "delta": _period_delta(df, kpi["column"], kpi.get("agg", "sum"), date_col),
                "raw": value,
            }
        )
    return cards


# --------------------------------------------------------------------------- #
# Charts
# --------------------------------------------------------------------------- #
def _build_figure(chart: dict, df: pd.DataFrame) -> tuple[go.Figure | None, pd.DataFrame | None]:
    ctype = chart["type"]
    x, y, color = chart.get("x"), chart.get("y"), chart.get("color")
    agg = chart.get("agg", "sum")
    top_n = chart.get("top_n", 10)

    if ctype in ("bar", "hbar", "pie"):
        table = _category_table(df, x, y, agg, top_n)
        if ctype == "pie":
            if len(table) > 5:
                ctype = "bar"
            else:
                fig = px.pie(table, names=x, values="value", title=chart["title"], hole=0.45)
                return _style(fig), table
        if ctype == "hbar":
            table = table.sort_values("value")
            fig = px.bar(
                table, x="value", y=x, orientation="h", title=chart["title"],
                labels={"value": pretty_label(y or "Count")},
            )
        else:
            fig = px.bar(
                table, x=x, y="value", title=chart["title"],
                labels={"value": pretty_label(y or "Count")},
            )
        return _style(fig), table

    if ctype == "line":
        table = _date_table(df, x, y, agg, color)
        fig = px.line(
            table, x=x, y="value", color=color, markers=True, title=chart["title"],
            labels={"value": pretty_label(y)},
        )
        return _style(fig), table

    if ctype == "stacked_bar":
        keys = [x] + ([color] if color else [])
        table = _grouped(df, keys, y, agg).reset_index()
        fig = px.bar(
            table, x=x, y="value", color=color, barmode="stack", title=chart["title"],
            labels={"value": pretty_label(y)},
        )
        return _style(fig), table

    if ctype == "scatter":
        data = df[[c for c in (x, y, color) if c]].dropna()
        if len(data) > MAX_SCATTER_POINTS:
            data = data.sample(MAX_SCATTER_POINTS, random_state=0)
        fig = px.scatter(data, x=x, y=y, color=color, title=chart["title"], opacity=0.7)
        return _style(fig), data

    if ctype == "histogram":
        data = df[[x]].dropna()
        fig = px.histogram(data, x=x, nbins=30, title=chart["title"])
        return _style(fig), data

    if ctype == "box":
        group = color or x
        data = df[[c for c in (group, y) if c]].dropna()
        fig = px.box(data, x=group, y=y, title=chart["title"])
        return _style(fig), data

    if ctype == "heatmap":
        pivot = pd.crosstab(df[x], df[y])
        fig = go.Figure(
            go.Heatmap(
                z=pivot.values,
                x=[str(c) for c in pivot.columns],
                y=[str(i) for i in pivot.index],
                colorscale="Blues",
            )
        )
        fig.update_layout(title=chart["title"], xaxis_title=pretty_label(x), yaxis_title=pretty_label(y))
        return _style(fig), pivot.reset_index()

    if ctype == "table":
        columns = [c for c in (x, y, color) if c]
        table = df[columns].head(MAX_TABLE_ROWS) if columns else df.head(MAX_TABLE_ROWS)
        return None, table

    raise ValueError(f"unsupported chart type {ctype}")


def build_dashboard(spec: dict, df: pd.DataFrame) -> dict:
    """Return KPI cards and rendered charts. Bad charts are skipped, not raised."""
    date_col = None
    for column in spec.get("filters", []):
        if column in df.columns and looks_like_date(df[column]):
            date_col = column
            break
    if date_col is None and spec.get("charts"):
        for chart in spec["charts"]:
            if chart["type"] == "line" and chart.get("x"):
                date_col = chart["x"]
                break

    result = {"kpis": kpi_cards(spec, df, date_col), "charts": []}

    for chart in spec.get("charts", []):
        try:
            figure, table = _build_figure(chart, df)
            value_fmt = "currency" if any(
                k["column"] == chart.get("y") and k.get("format") == "currency"
                for k in spec.get("kpis", [])
            ) else "number"
            result["charts"].append(
                {
                    "chart": chart,
                    "figure": figure,
                    "table": table,
                    "insight": _insight_for(chart, table, value_fmt),
                    "error": None,
                }
            )
        except Exception as exc:
            result["charts"].append(
                {
                    "chart": chart,
                    "figure": None,
                    "table": None,
                    "insight": "",
                    "error": f"{type(exc).__name__}: {exc}",
                }
            )
    return result


In [ ]:
%%writefile dashboard/export.py
"""Export a dashboard to JSON, CSV, Excel, HTML, PNG, PDF and Hyper.

Every function returns bytes and raises nothing the caller must handle:
the app checks :func:`availability` and shows a reason when a format is
unavailable instead of crashing.
"""
from __future__ import annotations

import io
import json
import re
import tempfile
import zipfile
from datetime import date, datetime
from pathlib import Path

import pandas as pd

from .profile import pretty_label
from .render import format_compact

_PNG_OK: bool | None = None


# --------------------------------------------------------------------------- #
# Availability
# --------------------------------------------------------------------------- #
def _can_import(module: str) -> bool:
    try:
        __import__(module)
        return True
    except Exception:
        return False


def png_supported() -> bool:
    """Render one tiny figure once; kaleido also needs a Chrome binary."""
    global _PNG_OK
    if _PNG_OK is None:
        try:
            import plotly.graph_objects as go

            probe = go.Figure(go.Bar(x=["a"], y=[1]))
            _PNG_OK = bool(probe.to_image(format="png"))
        except Exception:
            _PNG_OK = False
    return bool(_PNG_OK)


def availability() -> dict[str, tuple[bool, str]]:
    has_png = png_supported()
    return {
        "png": (has_png, "Needs kaleido and a Chrome/Chromium binary."),
        "pdf": (has_png and _can_import("fpdf"), "Needs fpdf2 and PNG rendering."),
        "html": (True, ""),
        "excel": (_can_import("openpyxl"), "Needs openpyxl."),
        "csv": (True, ""),
        "hyper": (
            _can_import("tableauhyperapi"),
            "Needs tableauhyperapi (x86_64 Linux/Windows only).",
        ),
        "json": (True, ""),
    }


# --------------------------------------------------------------------------- #
# Helpers
# --------------------------------------------------------------------------- #
def _safe_sheet_name(name: str, used: set[str]) -> str:
    cleaned = re.sub(r"[\[\]:*?/\\]", " ", str(name))[:31] or "Sheet"
    candidate, index = cleaned, 2
    while candidate in used:
        suffix = f" {index}"
        candidate = cleaned[: 31 - len(suffix)] + suffix
        index += 1
    used.add(candidate)
    return candidate


def _slug(text: str) -> str:
    return re.sub(r"[^a-z0-9]+", "-", str(text).lower()).strip("-") or "chart"


def _tables_as_list(charts: list[dict]) -> list[tuple[str, pd.DataFrame]]:
    tables = []
    for index, item in enumerate(charts, start=1):
        table = item.get("table")
        if isinstance(table, pd.DataFrame) and not table.empty:
            tables.append((f"Chart {index} - {item['chart'].get('title', '')}", table))
    return tables


# --------------------------------------------------------------------------- #
# Formats
# --------------------------------------------------------------------------- #
def to_json(spec: dict) -> bytes:
    return json.dumps(spec, indent=2, ensure_ascii=False).encode("utf-8")


def to_csv_zip(df: pd.DataFrame, charts: list[dict]) -> bytes:
    buffer = io.BytesIO()
    with zipfile.ZipFile(buffer, "w", zipfile.ZIP_DEFLATED) as archive:
        archive.writestr("data.csv", df.to_csv(index=False))
        for index, (name, table) in enumerate(_tables_as_list(charts), start=1):
            archive.writestr(f"chart-{index}-{_slug(name)}.csv", table.to_csv(index=False))
    return buffer.getvalue()


def to_excel(df: pd.DataFrame, charts: list[dict], kpis: list[dict], spec: dict) -> bytes:
    buffer = io.BytesIO()
    used: set[str] = set()
    summary = pd.DataFrame(
        [
            {
                "KPI": card["label"],
                "Value": card["value"],
                "Change vs previous period": (
                    f"{card['delta']:+.1f}%" if card.get("delta") is not None else ""
                ),
            }
            for card in kpis
        ]
    )
    with pd.ExcelWriter(buffer, engine="openpyxl") as writer:
        summary.to_excel(writer, sheet_name=_safe_sheet_name("Summary", used), index=False)
        df.to_excel(writer, sheet_name=_safe_sheet_name("Data", used), index=False)
        for name, table in _tables_as_list(charts):
            table.to_excel(writer, sheet_name=_safe_sheet_name(name, used), index=False)
    return buffer.getvalue()


def to_html(spec: dict, charts: list[dict], kpis: list[dict]) -> bytes:
    cards = "".join(
        f'<div class="kpi"><div class="label">{card["label"]}</div>'
        f'<div class="value">{card["value"]}</div>'
        + (
            f'<div class="delta">{card["delta"]:+.1f}% vs previous period</div>'
            if card.get("delta") is not None
            else ""
        )
        + "</div>"
        for card in kpis
    )

    blocks = []
    for index, item in enumerate(charts):
        if item.get("figure") is None:
            continue
        blocks.append(
            '<section class="chart">'
            + item["figure"].to_html(
                full_html=False,
                include_plotlyjs=True if not blocks else False,
                config={"displayModeBar": True, "responsive": True},
            )
            + f'<p class="insight">{item.get("insight", "")}</p>'
            + "</section>"
        )

    html = f"""<!doctype html>
<html lang="en"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>{spec.get("title", "Dashboard")}</title>
<style>
 body {{ font-family: system-ui, -apple-system, sans-serif; margin: 0; background: #f8fafc; color: #0f172a; }}
 header {{ padding: 32px 40px 8px; }}
 h1 {{ margin: 0 0 4px; font-size: 26px; }}
 .subtitle {{ color: #64748b; margin: 0; }}
 .kpis {{ display: flex; gap: 16px; padding: 24px 40px; flex-wrap: wrap; }}
 .kpi {{ background: #fff; border: 1px solid #e2e8f0; border-radius: 12px; padding: 16px 20px; min-width: 180px; }}
 .kpi .label {{ color: #64748b; font-size: 13px; }}
 .kpi .value {{ font-size: 26px; font-weight: 600; margin-top: 4px; }}
 .kpi .delta {{ color: #2563eb; font-size: 12px; margin-top: 4px; }}
 .chart {{ background: #fff; border: 1px solid #e2e8f0; border-radius: 12px; margin: 0 40px 20px; padding: 12px; }}
 .insight {{ color: #475569; font-size: 14px; margin: 4px 8px 8px; }}
 footer {{ color: #64748b; font-size: 12px; padding: 16px 40px 40px; }}
</style></head>
<body>
<header><h1>{spec.get("title", "Dashboard")}</h1><p class="subtitle">{spec.get("subtitle", "")}</p></header>
<div class="kpis">{cards}</div>
{"".join(blocks)}
<footer>Generated by AI from your data. Verify important numbers.</footer>
</body></html>"""
    return html.encode("utf-8")


def chart_png(fig) -> bytes | None:
    try:
        return fig.to_image(format="png", scale=2)
    except Exception:
        return None


def dashboard_png(pngs: list[bytes]) -> bytes | None:
    """Stack chart images vertically into one dashboard image."""
    images = [png for png in pngs if png]
    if not images:
        return None
    try:
        from PIL import Image

        loaded = [Image.open(io.BytesIO(png)).convert("RGB") for png in images]
        width = max(image.width for image in loaded)
        gap = 24
        height = sum(image.height for image in loaded) + gap * (len(loaded) - 1)
        canvas = Image.new("RGB", (width, height), "white")
        offset = 0
        for image in loaded:
            canvas.paste(image, ((width - image.width) // 2, offset))
            offset += image.height + gap
        buffer = io.BytesIO()
        canvas.save(buffer, format="PNG")
        return buffer.getvalue()
    except Exception:
        return None


def _latin(text: str) -> str:
    return str(text).encode("latin-1", "replace").decode("latin-1")


def to_pdf(spec: dict, kpis: list[dict], pngs: list[bytes]) -> bytes | None:
    try:
        from fpdf import FPDF
    except Exception:
        return None

    pdf = FPDF(format="A4")
    pdf.set_auto_page_break(auto=True, margin=12)
    pdf.add_page()

    pdf.set_font("Helvetica", "B", 18)
    pdf.cell(0, 10, _latin(spec.get("title", "Dashboard")), new_x="LMARGIN", new_y="NEXT")
    pdf.set_font("Helvetica", "", 10)
    pdf.set_text_color(100, 116, 139)
    pdf.multi_cell(0, 5, _latin(spec.get("subtitle", "")))
    pdf.ln(4)

    pdf.set_text_color(15, 23, 42)
    for card in kpis:
        pdf.set_font("Helvetica", "B", 12)
        pdf.cell(60, 8, _latin(f"{card['label']}: {card['value']}"), new_x="LMARGIN", new_y="NEXT")
    pdf.ln(2)

    for png in pngs:
        if not png:
            continue
        try:
            pdf.image(io.BytesIO(png), w=180)
            pdf.ln(4)
        except Exception:
            continue

    pdf.set_font("Helvetica", "I", 8)
    pdf.set_text_color(100, 116, 139)
    pdf.multi_cell(0, 4, "Generated by AI from your data. Verify important numbers.")
    return bytes(pdf.output())


def to_hyper(df: pd.DataFrame) -> bytes | None:
    """Write a Tableau Hyper extract (x86_64 Linux/Windows wheels only)."""
    try:
        from tableauhyperapi import (
            Connection,
            CreateMode,
            HyperProcess,
            Inserter,
            SqlType,
            TableDefinition,
            Telemetry,
        )
    except Exception:
        return None

    def sql_type(series: pd.Series):
        if pd.api.types.is_datetime64_any_dtype(series):
            return SqlType.date()
        if pd.api.types.is_bool_dtype(series):
            return SqlType.bool()
        if pd.api.types.is_numeric_dtype(series):
            return SqlType.double()
        return SqlType.text()

    def value(item):
        if item is None or (isinstance(item, float) and pd.isna(item)):
            return None
        if isinstance(item, (pd.Timestamp, datetime)):
            return item.date()
        if isinstance(item, (int, float)) and not isinstance(item, bool):
            return float(item)
        return str(item)

    with tempfile.TemporaryDirectory() as tmp:
        path = Path(tmp) / "extract.hyper"
        with HyperProcess(Telemetry.DO_NOT_SEND_USAGE_DATA_TO_TABLEAU) as hyper:
            with Connection(hyper.endpoint, str(path), CreateMode.CREATE_AND_REPLACE) as conn:
                definition = TableDefinition(
                    "Extract",
                    [
                        TableDefinition.Column(str(col), sql_type(df[col]))
                        for col in df.columns
                    ],
                )
                conn.catalog.create_table(definition)
                rows = [tuple(value(v) for v in row) for row in df.itertuples(index=False)]
                with Inserter(conn, definition) as inserter:
                    inserter.add_rows(rows)
                    inserter.execute()
        return path.read_bytes()


def to_tableau_notes() -> str:
    return (
        "Tableau and Power BI cannot import a finished Plotly layout. "
        "These exports carry the data; rebuild the visuals there."
    )


In [ ]:
%%writefile tests/test_dashboard.py
"""Tests for dashboard profiling, spec validation, fallback and rendering."""
import sys
from pathlib import Path

import pandas as pd
import pytest

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))

from dashboard import planner, profile, render  # noqa: E402


@pytest.fixture
def df() -> pd.DataFrame:
    rows = 40
    return pd.DataFrame(
        {
            "order_id": [f"ORD-{i}" for i in range(rows)],
            "date": pd.date_range("2026-01-01", periods=rows, freq="D"),
            "product": ["A", "B", "C", "D", "E", "F"] * 6 + ["A"] * 4,
            "region": ["North", "South"] * 20,
            "revenue": [i * 10.0 for i in range(rows)],
            "quantity": [i % 5 for i in range(rows)],
        }
    )


def test_profile_detects_kinds_and_ids(df):
    prof = profile.profile_dataframe(df)
    kinds = {col["name"]: col["kind"] for col in prof["columns"]}
    ids = {col["name"]: col["is_id"] for col in prof["columns"]}

    assert kinds["date"] == "date"
    assert kinds["revenue"] == "numeric"
    assert kinds["product"] == "categorical"
    assert ids["order_id"] is True
    assert prof["date_range"]["column"] == "date"
    assert prof["rows"] == 40


def test_parse_json_tolerates_markdown_fences():
    spec = planner.parse_json('```json\n{"title": "x", "charts": []}\n```')
    assert spec["title"] == "x"


def test_validate_drops_unknown_columns(df):
    prof = profile.profile_dataframe(df)
    spec = {
        "title": "T",
        "charts": [{"type": "bar", "x": "missing_col", "y": "revenue", "agg": "sum"}],
        "kpis": [],
    }
    clean, dropped = planner.validate_spec(spec, prof)

    assert clean["charts"] == []
    assert dropped
    assert "missing_col" in dropped[0]["reason"]


def test_validate_drops_incompatible_chart_type(df):
    prof = profile.profile_dataframe(df)
    spec = {
        "charts": [{"type": "line", "x": "product", "y": "revenue"}],
        "kpis": [],
    }
    clean, dropped = planner.validate_spec(spec, prof)

    assert clean["charts"] == []
    assert "date x" in dropped[0]["reason"]


def test_unique_numeric_measure_is_not_an_id(df):
    prof = profile.profile_dataframe(df)
    ids = {col["name"]: col["is_id"] for col in prof["columns"]}

    assert ids["order_id"] is True
    assert ids["revenue"] is False
    assert "revenue" in profile.usable_columns(prof, "numeric")


def test_swapped_axes_are_repaired(df):
    prof = profile.profile_dataframe(df)
    spec = {
        "charts": [{"type": "hbar", "x": "revenue", "y": "product", "agg": "sum"}],
        "kpis": [],
    }
    clean, dropped = planner.validate_spec(spec, prof)

    assert dropped == []
    assert clean["charts"][0]["x"] == "product"
    assert clean["charts"][0]["y"] == "revenue"


def test_pie_is_downgraded_when_too_many_slices(df):
    prof = profile.profile_dataframe(df)
    spec = {
        "charts": [{"type": "pie", "x": "product", "y": "revenue", "agg": "sum"}],
        "kpis": [],
    }
    clean, _ = planner.validate_spec(spec, prof)

    assert len(clean["charts"]) == 1
    assert clean["charts"][0]["type"] == "bar"


def test_chart_count_is_capped(df):
    prof = profile.profile_dataframe(df)
    spec = {
        "charts": [
            {"type": "bar", "x": "product", "y": "revenue", "agg": "sum"}
            for _ in range(10)
        ],
        "kpis": [],
    }
    clean, dropped = planner.validate_spec(spec, prof)

    assert len(clean["charts"]) == planner.MAX_CHARTS
    assert any("6-chart limit" in item["reason"] for item in dropped)


def test_filters_must_be_categorical_or_date(df):
    prof = profile.profile_dataframe(df)
    clean, dropped = planner.validate_spec(
        {"charts": [], "kpis": [], "filters": ["revenue", "region", "date", "product", "quantity"]},
        prof,
    )

    assert clean["filters"] == ["region", "date", "product"]
    assert any(item["item"] == "revenue" for item in dropped)


def test_fallback_spec_is_usable(df):
    prof = profile.profile_dataframe(df)
    spec = planner.fallback_spec(prof)

    assert spec["charts"]
    assert len(spec["charts"]) <= planner.MAX_CHARTS
    assert spec["kpis"]

    names = {col["name"] for col in prof["columns"]}
    for chart in spec["charts"]:
        for key in ("x", "y", "color"):
            assert chart[key] is None or chart[key] in names


def test_render_builds_figures_and_insights(df):
    prof = profile.profile_dataframe(df)
    spec = planner.fallback_spec(prof)

    result = render.build_dashboard(spec, df)

    assert result["kpis"]
    assert result["charts"]
    assert any(item["figure"] is not None for item in result["charts"])
    assert all(item["error"] is None for item in result["charts"])
    assert any(item["insight"] for item in result["charts"])


def test_filters_apply_locally(df):
    filtered = render.apply_filters(df, {"region": ["North"]})
    assert set(filtered["region"]) == {"North"}
    assert len(filtered) < len(df)


def test_insight_uses_real_numbers(df):
    prof = profile.profile_dataframe(df)
    spec = {
        "title": "T",
        "subtitle": "",
        "kpis": [],
        "charts": [{"type": "hbar", "x": "product", "y": "revenue", "agg": "sum", "insight": ""}],
        "filters": [],
    }
    clean, _ = planner.validate_spec(spec, prof)
    result = render.build_dashboard(clean, df)

    insight = result["charts"][0]["insight"]
    assert insight
    assert "leads with" in insight


## Layer 9: User Interface

`app.py` - a Streamlit front end with two tabs. The user uploads a CSV or Excel file, asks questions in plain English, and gets answers with interactive Plotly charts - or switches to the Dashboard tab for a generated dashboard. No code is shown unless they ask for it.

The app reuses `src/agent` unchanged: it calls `ask()` and reads the JSONL trace that the observability layer already writes, so no agent logic is duplicated.

- Sidebar: upload (CSV/XLSX), data preview, starter questions, dashboard filters, clear conversation, "show the code" toggle
- Chat tab: interactive charts, PNG/HTML download, and a collapsed "How I got this" section
- Dashboard tab: generated KPIs and charts, local filters, customize, data quality, and an export panel
- XLSX files are converted to CSV in the background and offered as a download
- Errors and step-limit stops are handled with a friendly message, never a traceback


In [ ]:
%%writefile app.py
"""Streamlit interface for the Data Analysis Agent.

Two tabs:
  Chat      - ask questions, the agent writes and runs code
  Dashboard - the agent plans a JSON spec, the app renders it

Run locally:    streamlit run app.py
Run in Docker:  docker compose up ui   (then open http://localhost:8501)
"""
from __future__ import annotations

import io
import json
import sys
import zipfile
from pathlib import Path

import pandas as pd
import streamlit as st

st.set_page_config(
    page_title="Data Analysis Agent",
    page_icon="📊",
    layout="wide",
    initial_sidebar_state="expanded",
)

ROOT = Path(__file__).parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "src"))

try:
    from agent.agent import ask
    from agent.tools import FIGURES

    from dashboard import export as dash_export
    from dashboard import planner as dash_planner
    from dashboard import profile as dash_profile
    from dashboard import render as dash_render

    AGENT_READY, AGENT_IMPORT_ERROR = True, None
except Exception as exc:  # missing key, missing package, ...
    AGENT_READY, AGENT_IMPORT_ERROR = False, exc

LOG_PATH = ROOT / "logs" / "agent.jsonl"
MAX_UPLOAD_MB = 25
STEP_LIMIT_TEXT = "Stopped: hit step limit."
FRIENDLY_ERROR = (
    "I couldn't answer that reliably. Try rephrasing the question "
    "or asking something narrower."
)
TRUST_NOTE = "Answers are generated by AI from your data. Check important numbers."
EXPORT_LABELS = {
    "png": "PNG images",
    "pdf": "PDF (one page)",
    "html": "HTML (interactive)",
    "excel": "Excel (.xlsx)",
    "csv": "CSV (zip)",
    "hyper": "Tableau (.hyper)",
    "json": "Dashboard spec (.json)",
}


# --------------------------------------------------------------------------- #
# Data loading
# --------------------------------------------------------------------------- #
@st.cache_data(show_spinner=False)
def _parse_table(raw: bytes, name: str) -> pd.DataFrame:
    if name.lower().endswith((".xlsx", ".xlsm")):
        return pd.read_excel(io.BytesIO(raw), engine="openpyxl")
    return pd.read_csv(io.BytesIO(raw))


def load_table(raw: bytes, name: str) -> pd.DataFrame:
    # The agent's generated code can mutate df (e.g. parsing dates), so never
    # hand out the cached object itself.
    return _parse_table(raw, name).copy()


# --------------------------------------------------------------------------- #
# Chat helpers
# --------------------------------------------------------------------------- #
def data_summary(df: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame(
        {
            "column": df.columns.astype(str),
            "type": [str(t) for t in df.dtypes],
            "missing": df.isna().sum().to_numpy(),
        }
    )


def _looks_like_date(series: pd.Series) -> bool:
    if pd.api.types.is_datetime64_any_dtype(series):
        return True
    name = str(series.name).lower()
    return not pd.api.types.is_numeric_dtype(series) and any(
        key in name for key in ("date", "time", "month", "year", "day")
    )


def _pick_metric(numeric_cols: list[str]) -> str:
    for key in ("revenue", "sales", "amount", "total", "price", "value", "cost"):
        for col in numeric_cols:
            if key in col.lower():
                return col
    return numeric_cols[-1]


def _is_identifier(df: pd.DataFrame, col: str) -> bool:
    """IDs, codes and free-text keys make poor grouping columns."""
    name = str(col).lower()
    if name in {"id", "index", "uuid"} or name.endswith(("_id", "id", "_no", "code")):
        return True
    return df[col].nunique(dropna=True) == len(df) > 20


def suggest_questions(df: pd.DataFrame) -> list[str]:
    """Starter questions derived from the uploaded columns."""
    numeric, categorical, temporal = [], [], []
    for col in df.columns:
        series = df[col]
        if pd.api.types.is_numeric_dtype(series):
            numeric.append(col)
        elif _looks_like_date(series):
            temporal.append(col)
        elif not _is_identifier(df, col):
            categorical.append(col)

    # Prefer business-like dimensions (in priority order), then fewest distinct values.
    preferred = ("product", "item", "category", "region", "segment", "customer", "name", "type")

    def rank(col: str) -> tuple[int, int]:
        name = str(col).lower()
        for index, key in enumerate(preferred):
            if key in name:
                return (index, df[col].nunique(dropna=True))
        return (len(preferred), df[col].nunique(dropna=True))

    categorical.sort(key=rank)

    questions: list[str] = []
    if categorical and numeric:
        metric = _pick_metric(numeric)
        questions.append(f"Which {categorical[0]} has the lowest {metric}?")
        questions.append(f"Which {categorical[0]} has the highest {metric}?")
    if temporal and numeric:
        questions.append(f"Is there a trend in {_pick_metric(numeric)} over time?")
    if numeric:
        questions.append(f"Are there outliers in {_pick_metric(numeric)}?")
    questions.append("Are there missing values I should worry about?")
    return questions[:4]


def _log_size() -> int:
    try:
        return LOG_PATH.stat().st_size
    except OSError:
        return 0


def _read_new_trace(offset: int) -> dict | None:
    """The agent logs one JSON line per completed question; read what was appended."""
    try:
        with LOG_PATH.open("rb") as handle:
            handle.seek(offset)
            chunk = handle.read().decode("utf-8", "ignore")
    except OSError:
        return None
    for line in reversed(chunk.splitlines()):
        line = line.strip()
        if not line:
            continue
        try:
            return json.loads(line)
        except json.JSONDecodeError:
            continue
    return None


def run_agent(question: str, df: pd.DataFrame, history):
    """Call the existing agent, returning its answer plus the trace and charts."""
    offset = _log_size()
    figures_before = len(FIGURES)

    # Work on a copy so a failed call cannot leave a dangling user message in history.
    working = list(history) if history else None

    error: str | None = None
    try:
        answer, new_history = ask(question, df, working)
    except Exception as exc:  # network, auth, rate limit...
        answer, new_history, error = None, history, f"{type(exc).__name__}: {exc}"

    trace = _read_new_trace(offset) or {}
    steps = trace.get("steps", [])
    figures = FIGURES[figures_before:]
    return answer, new_history, steps, figures, error, trace.get("tokens")


def chart_png(fig) -> bytes | None:
    try:
        return fig.to_image(format="png", scale=2)
    except Exception:
        return None


def conversation_markdown() -> str:
    lines = ["# Data Analysis Agent - conversation", ""]
    for entry in st.session_state.chat:
        if entry["role"] == "user":
            lines += ["## You", "", entry["content"], ""]
            continue
        lines += ["## Agent", "", entry["answer"], ""]
        for step in entry.get("steps", []):
            lines += ["```python", step.get("code", ""), "```", ""]
            result = (step.get("result") or "").strip()
            if result:
                lines += ["```text", result[:2000], "```", ""]
    return "\n".join(lines)


def _next_id() -> int:
    st.session_state.entry_seq += 1
    return st.session_state.entry_seq


def render_assistant(entry: dict, show_code: bool) -> None:
    st.markdown(entry["answer"])

    if "png" not in entry:
        entry["png"] = [chart_png(fig) for fig in entry.get("figures", [])]

    for index, fig in enumerate(entry.get("figures", [])):
        st.plotly_chart(fig, width="stretch", key=f"chart-{entry['id']}-{index}")
        png = entry["png"][index] if index < len(entry["png"]) else None
        if png:
            st.download_button(
                "Download chart as PNG",
                png,
                file_name=f"chart-{index + 1}.png",
                mime="image/png",
                key=f"png-{entry['id']}-{index}",
            )
        else:
            st.download_button(
                "Download chart (.html)",
                fig.to_html(include_plotlyjs="cdn"),
                file_name=f"chart-{index + 1}.html",
                mime="text/html",
                key=f"html-{entry['id']}-{index}",
            )

    steps = entry.get("steps") or []
    if not steps and not entry.get("error"):
        return

    with st.expander("How I got this", expanded=show_code):
        if entry.get("error"):
            st.caption("The agent raised an error on this question:")
            st.code(entry["error"], language="text")
        for step in steps:
            flags = "error" if step.get("error") else "ok"
            st.caption(f"{step.get('latency_s', '?')}s · {flags}")
            st.code(step.get("code", ""), language="python")
            result = (step.get("result") or "").strip()
            if result:
                st.text(result[:2000])


# --------------------------------------------------------------------------- #
# Dashboard helpers
# --------------------------------------------------------------------------- #
def _is_date_series(df: pd.DataFrame, column: str) -> bool:
    return dash_profile.looks_like_date(df[column])


def dashboard_filters(spec: dict, df: pd.DataFrame) -> dict:
    """Sidebar filters. Purely local: changing them never calls the model."""
    values: dict = {}
    with st.sidebar:
        st.divider()
        st.subheader("Dashboard filters")
        for column in spec.get("filters", [])[:3]:
            if column not in df.columns:
                continue
            label = dash_profile.pretty_label(column)
            if _is_date_series(df, column):
                parsed = pd.to_datetime(df[column], errors="coerce").dropna()
                if parsed.empty:
                    continue
                low, high = parsed.min().date(), parsed.max().date()
                picked = st.date_input(
                    label, value=(low, high), min_value=low, max_value=high, key=f"flt-{column}"
                )
                if isinstance(picked, tuple) and len(picked) == 2:
                    values[column] = picked
            else:
                options = sorted(df[column].dropna().astype(str).unique())
                picked = st.multiselect(label, options, key=f"flt-{column}")
                if picked:
                    values[column] = picked
        if values:
            st.caption("Filters apply locally and do not use API credits.")
    return values


def apply_overrides(spec: dict, overrides: dict) -> dict:
    charts = []
    for index, chart in enumerate(spec.get("charts", [])):
        override = overrides.get(index)
        if override and not override.get("keep", True):
            continue
        chart = dict(chart)
        if override and override.get("type"):
            chart["type"] = override["type"]
        charts.append(chart)
    return {**spec, "charts": charts}


def data_quality(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    missing = df.isna().sum()
    missing = missing[missing > 0].sort_values(ascending=False)
    missing_table = pd.DataFrame(
        {"column": missing.index.astype(str), "missing": missing.to_numpy()}
    )

    rows = []
    for column in df.columns:
        series = df[column]
        if not pd.api.types.is_numeric_dtype(series):
            continue
        clean = series.dropna()
        if len(clean) < 10:
            continue
        q1, q3 = clean.quantile([0.25, 0.75])
        iqr = q3 - q1
        if iqr == 0:
            continue
        count = int(((clean < q1 - 1.5 * iqr) | (clean > q3 + 1.5 * iqr)).sum())
        if count:
            rows.append({"column": str(column), "outliers (IQR)": count})
    return missing_table, pd.DataFrame(rows)


def _render_chart(item: dict, key: str) -> None:
    chart = item["chart"]
    st.markdown(f"**{chart.get('title', 'Chart')}**")
    if item.get("figure") is not None:
        st.plotly_chart(item["figure"], width="stretch", key=key)
    elif item.get("table") is not None:
        st.dataframe(item["table"], width="stretch", hide_index=True)
    if item.get("insight"):
        st.caption(item["insight"])
    if item.get("error"):
        st.caption(f"Skipped: {item['error']}")


def _build_export_files(spec, filtered, rendered, kpis) -> tuple[dict, dict]:
    """Build every export in memory. Returns (files, reasons)."""
    files: dict[str, tuple[bytes, str, str]] = {}
    reasons: dict[str, str] = {}
    available = dash_export.availability()

    def attempt(key, builder, filename, mime):
        if not available.get(key, (False, ""))[0]:
            reasons[key] = available.get(key, (False, "Unavailable."))[1]
            return
        try:
            payload = builder()
            if payload:
                files[key] = (payload, filename, mime)
            else:
                reasons[key] = "Nothing to export."
        except Exception as exc:
            reasons[key] = f"{type(exc).__name__}: {exc}"

    attempt("json", lambda: dash_export.to_json(spec), "dashboard.json", "application/json")
    attempt("csv", lambda: dash_export.to_csv_zip(filtered, rendered), "dashboard-data.zip", "application/zip")
    attempt(
        "excel",
        lambda: dash_export.to_excel(filtered, rendered, kpis, spec),
        "dashboard.xlsx",
        "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet",
    )
    attempt("html", lambda: dash_export.to_html(spec, rendered, kpis), "dashboard.html", "text/html")

    pngs: list[bytes] = []
    if available["png"][0]:
        for item in rendered:
            if item.get("figure") is not None:
                png = dash_export.chart_png(item["figure"])
                if png:
                    pngs.append(png)

        def png_zip():
            if not pngs:
                return None
            buffer = io.BytesIO()
            with zipfile.ZipFile(buffer, "w", zipfile.ZIP_DEFLATED) as archive:
                for index, png in enumerate(pngs, start=1):
                    archive.writestr(f"chart-{index}.png", png)
                full = dash_export.dashboard_png(pngs)
                if full:
                    archive.writestr("dashboard.png", full)
            return buffer.getvalue()

        attempt("png", png_zip, "dashboard-charts.zip", "application/zip")
        attempt(
            "pdf",
            lambda: dash_export.to_pdf(spec, kpis, pngs),
            "dashboard.pdf",
            "application/pdf",
        )
    else:
        reasons["png"] = available["png"][1]
        reasons["pdf"] = available["pdf"][1]

    attempt("hyper", lambda: dash_export.to_hyper(filtered), "dashboard.hyper", "application/octet-stream")
    return files, reasons


def render_dashboard_tab(df: pd.DataFrame | None) -> None:
    st.caption(TRUST_NOTE)
    if df is None:
        st.info("Upload a CSV to build a dashboard")
        return

    state = st.session_state.dashboard
    focus = st.text_input(
        "Focus on... (optional)",
        value=st.session_state.dashboard_focus,
        placeholder="e.g. regional performance",
        key="dashboard_focus_input",
    )

    label = "Regenerate" if state else "Generate dashboard"
    if st.button(label, type="primary"):
        st.session_state.dashboard_focus = focus
        with st.status("Profiling data...", expanded=True) as status:
            profile = dash_profile.profile_dataframe(df)
            status.update(label="Planning dashboard...")
            try:
                spec, meta = dash_planner.plan_dashboard(profile, focus)
            except Exception as exc:
                spec = dash_planner.fallback_spec(profile)
                meta = {
                    "fallback": True,
                    "dropped": [],
                    "tokens": 0,
                    "error": f"{type(exc).__name__}: {exc}",
                }
            status.update(label="Building charts...")
            st.session_state.dashboard = {
                "spec": spec,
                "meta": meta,
                "profile": profile,
                "overrides": {},
            }
            st.session_state.export_files = {}
            status.update(label="Dashboard ready", state="complete")
        st.rerun()

    if not state:
        st.info("Click **Generate dashboard** to build one from your data.")
        return

    spec = apply_overrides(state["spec"], state["overrides"])
    clean, dropped = dash_planner.validate_spec(spec, state["profile"])
    spec = clean or state["spec"]

    filters = dashboard_filters(spec, df)
    filtered = dash_render.apply_filters(df, filters)

    st.divider()
    st.markdown(f"### {spec['title']}")
    st.caption(spec["subtitle"])

    profile = state["profile"]
    span = profile.get("date_range")
    freshness = f"{len(filtered):,} of {profile['rows']:,} rows"
    if span:
        freshness += f" · {span['min']} to {span['max']}"
    st.caption(freshness)

    if filtered.empty:
        st.warning("No rows match the current filters.")
        return

    rendered = dash_render.build_dashboard(spec, filtered)
    kpis = rendered["kpis"]
    charts = rendered["charts"]

    if dropped:
        st.caption(f"Skipped {len(dropped)} chart(s) that did not fit the data.")
    if state["meta"].get("fallback"):
        st.caption("The planner returned an unusable spec, so this is the rule-based default dashboard.")

    if kpis:
        columns = st.columns(len(kpis))
        for column, card in zip(columns, kpis):
            delta = f"{card['delta']:+.1f}% vs prev." if card.get("delta") is not None else None
            column.metric(card["label"], card["value"], delta=delta)

    if charts:
        _render_chart(charts[0], key="dash-main")
        grid = charts[1:]
        for index in range(0, len(grid), 2):
            left, right = st.columns(2)
            with left:
                _render_chart(grid[index], key=f"dash-{index}")
            if index + 1 < len(grid):
                with right:
                    _render_chart(grid[index + 1], key=f"dash-{index + 1}")

    missing_table, outliers = data_quality(filtered)
    with st.expander("Data quality", expanded=False):
        if missing_table.empty:
            st.caption("No missing values in the filtered data.")
        else:
            st.caption("Missing values per column")
            st.dataframe(missing_table, width="stretch", hide_index=True)
        if outliers.empty:
            st.caption("No outliers detected (1.5 × IQR rule).")
        else:
            st.caption("Detected outliers (1.5 × IQR rule)")
            st.dataframe(outliers, width="stretch", hide_index=True)

    with st.expander("Customize", expanded=False):
        overrides = {}
        for index, chart in enumerate(state["spec"].get("charts", [])):
            left, right = st.columns([3, 2])
            keep = left.checkbox(
                chart.get("title", f"Chart {index + 1}"),
                value=state["overrides"].get(index, {}).get("keep", True),
                key=f"keep-{index}",
            )
            current = state["overrides"].get(index, {}).get("type", chart["type"])
            options = [chart["type"]] + [t for t in dash_planner.ALLOWED_TYPES if t != chart["type"]]
            new_type = right.selectbox(
                "Type", options, index=options.index(current), key=f"type-{index}",
                label_visibility="collapsed",
            )
            overrides[index] = {"keep": keep, "type": new_type}
        if st.button("Apply changes"):
            state["overrides"] = overrides
            st.rerun()

    with st.expander("Export", expanded=False):
        st.caption(dash_export.to_tableau_notes())
        available = dash_export.availability()
        if st.button("Prepare export files"):
            with st.spinner("Building files..."):
                files, reasons = _build_export_files(spec, filtered, charts, kpis)
            st.session_state.export_files = files
            st.session_state.export_reasons = reasons

        files = st.session_state.get("export_files", {})
        reasons = st.session_state.get("export_reasons", {})
        for key, label_text in EXPORT_LABELS.items():
            ok, reason = available.get(key, (False, "Unavailable."))
            if key in files:
                payload, filename, mime = files[key]
                st.download_button(
                    label_text, payload, file_name=filename, mime=mime, key=f"dl-{key}"
                )
            else:
                detail = reasons.get(key) or (reason if not ok else "Click Prepare export files.")
                st.button(
                    f"{label_text} — {detail}",
                    disabled=True,
                    key=f"disabled-{key}",
                )

    st.caption("Generated by AI from your data. Verify important numbers.")


# --------------------------------------------------------------------------- #
# Session state
# --------------------------------------------------------------------------- #
st.session_state.setdefault("chat", [])
st.session_state.setdefault("history", None)
st.session_state.setdefault("df", None)
st.session_state.setdefault("file_id", None)
st.session_state.setdefault("entry_seq", 0)
st.session_state.setdefault("pending_question", None)
st.session_state.setdefault("show_code", False)
st.session_state.setdefault("dashboard", None)
st.session_state.setdefault("dashboard_focus", "")
st.session_state.setdefault("export_files", {})
st.session_state.setdefault("export_reasons", {})

df: pd.DataFrame | None = st.session_state.df


# --------------------------------------------------------------------------- #
# Sidebar
# --------------------------------------------------------------------------- #
with st.sidebar:
    st.title("📊 Data Analysis Agent")
    st.caption("Upload a CSV or Excel file, ask questions, or generate a dashboard.")

    if not AGENT_READY:
        st.error(
            "The agent could not start. Check that GROQ_API_KEY is set in `.env` "
            "and that dependencies are installed."
        )
        with st.expander("Details"):
            st.code(f"{type(AGENT_IMPORT_ERROR).__name__}: {AGENT_IMPORT_ERROR}")
        st.stop()

    uploaded = st.file_uploader("Upload a CSV or Excel file", type=["csv", "xlsx"])

    if uploaded is not None:
        file_id = (uploaded.name, uploaded.size)
        if st.session_state.file_id != file_id:
            try:
                st.session_state.df = load_table(uploaded.getvalue(), uploaded.name)
                st.session_state.file_id = file_id
                st.session_state.chat = []
                st.session_state.history = None
                st.session_state.dashboard = None
                st.session_state.export_files = {}
                st.session_state.csv_bytes = st.session_state.df.to_csv(index=False).encode("utf-8")
                st.session_state.csv_name = Path(uploaded.name).stem + ".csv"
                df = st.session_state.df
            except Exception as exc:
                st.error(f"Could not read that file: {exc}")
                st.session_state.df = None
                df = None

        if uploaded.size > MAX_UPLOAD_MB * 1024 * 1024:
            st.warning(
                f"That file is {uploaded.size / 1e6:.0f} MB. Only a summary and a "
                "few sample rows are sent to the model, but large files may be slow."
            )

        if uploaded.name.lower().endswith(".xlsx") and st.session_state.get("csv_bytes"):
            st.caption(f"Converted to `{st.session_state['csv_name']}` in the background.")
            st.download_button(
                "Download converted CSV",
                st.session_state["csv_bytes"],
                file_name=st.session_state["csv_name"],
                mime="text/csv",
                width="stretch",
            )

    if df is not None:
        st.subheader("Data preview")
        rows, cols = df.shape
        left, right = st.columns(2)
        left.metric("Rows", f"{rows:,}")
        right.metric("Columns", cols)
        st.dataframe(
            data_summary(df),
            hide_index=True,
            width="stretch",
            height=min(260, 40 + 35 * len(df.columns)),
        )

        st.subheader("Try asking")
        for question in suggest_questions(df):
            if st.button(question, width="stretch", key=f"starter-{question}"):
                st.session_state.pending_question = question
                st.rerun()

        st.divider()

    st.toggle(
        "Show the code the agent wrote",
        key="show_code",
        help="Expands the 'How I got this' section under every answer.",
    )

    if st.button("Clear conversation", width="stretch", disabled=not st.session_state.chat):
        st.session_state.chat = []
        st.session_state.history = None
        st.session_state.pending_question = None
        st.rerun()

    if st.session_state.chat:
        st.download_button(
            "Download conversation (.md)",
            conversation_markdown(),
            file_name="data-analysis-conversation.md",
            mime="text/markdown",
            width="stretch",
        )

    st.caption(TRUST_NOTE)


# --------------------------------------------------------------------------- #
# Tabs
# --------------------------------------------------------------------------- #
tab_chat, tab_dashboard = st.tabs(["💬 Chat", "📊 Dashboard"])

with tab_chat:
    st.subheader("Conversation")

    if df is None:
        st.info("Upload a CSV in the sidebar to get started.")
    else:
        st.caption(
            f"Working with **{df.shape[0]:,} rows × {df.shape[1]} columns**. "
            "The model only receives a summary and three sample rows."
        )

    for entry in st.session_state.chat:
        if entry["role"] == "user":
            with st.chat_message("user"):
                st.markdown(entry["content"])
        else:
            with st.chat_message("assistant"):
                render_assistant(entry, st.session_state.show_code)

    prompt = st.chat_input(
        "Ask a question about your data...",
        disabled=df is None or not AGENT_READY,
    )

    if not prompt:
        prompt = st.session_state.pending_question
        st.session_state.pending_question = None

    if prompt and df is not None:
        st.session_state.chat.append({"role": "user", "content": prompt})
        with st.chat_message("user"):
            st.markdown(prompt)

        with st.chat_message("assistant"):
            with st.status("Analysing your data...", expanded=False) as status:
                status.update(label="Writing and running code...")
                answer, new_history, steps, figures, error, tokens = run_agent(
                    prompt, df, st.session_state.history
                )

                if error:
                    status.update(label="Something went wrong", state="error")
                elif any(step.get("error") for step in steps):
                    status.update(label="Recovered from an error", state="complete")
                elif figures:
                    status.update(label="Preparing chart...", state="complete")
                else:
                    status.update(label="Done", state="complete")

            if error:
                text = FRIENDLY_ERROR
            elif not answer or answer.strip() == STEP_LIMIT_TEXT:
                text = FRIENDLY_ERROR
            else:
                text = answer
                st.session_state.history = new_history

            entry = {
                "role": "assistant",
                "id": _next_id(),
                "answer": text,
                "steps": steps,
                "figures": figures,
                "error": error,
                "tokens": tokens,
            }
            st.session_state.chat.append(entry)
            render_assistant(entry, st.session_state.show_code)

with tab_dashboard:
    render_dashboard_tab(df)


In [21]:
import sys; sys.path.insert(0, "src")
import pandas as pd
from agent.agent import ask

df = pd.read_csv("data/sales.csv")
answer, history = ask("Which 5 products sold worst last quarter?", df)
print(answer)
answer, history = ask("Now break that down by region", df, history)   # follow-up keeps context

{"question": "Which 5 products sold worst last quarter?", "steps": [{"code": "\nimport pandas as pd\n# ensure date column is datetime\ndf['date'] = pd.to_datetime(df['date'])\n# Find max date\nmax_date = df['date'].max()\n# Determine quarter start\nquarter = (max_date.month - 1) // 3 + 1\n# quarter start month\nstart_month = 3 * (quarter - 1) + 1\n# quarter start date\nquarter_start = pd.Timestamp(year=max_date.year, month=start_month, day=1)\n# Filter last quarter dates (including up to max_date)\nmask = (df['date'] >= quarter_start) & (df['date'] <= max_date)\nlast_q = df.loc[mask]\n# Group by product sum quantity\nprod_qty = last_q.groupby('product')['quantity'].sum().reset_index()\n# Sort ascending\nworst5 = prod_qty.nsmallest(5, 'quantity')\nprint('Max date:', max_date)\nprint('Quarter start:', quarter_start)\nprint(worst5)\n", "result": "Max date: 2026-11-14 00:00:00\nQuarter start: 2026-10-01 00:00:00\n               product  quantity\n1    Bluetooth Speaker         4\n5        

**5 products with the lowest sales (by quantity) in the last quarter (Oct 1 – Nov 14, 2026)**  

| Rank | Product              | Units sold |
|------|----------------------|-----------|
| 1    | Bluetooth Speaker    | 4 |
| 2    | Standing Desk        | 6 |
| 3    | 27in Monitor         | 8 |
| 4    | Mechanical Keyboard  | 9 |
| 5    | Laptop Stand         | 10 |

These are the five products that sold the fewest units during the most recent quarter in the dataset.


{"question": "Now break that down by region", "steps": [{"code": "import pandas as pd\ndf['date'] = pd.to_datetime(df['date'])\nmax_date = df['date'].max()\nquarter = (max_date.month - 1) // 3 + 1\nstart_month = 3 * (quarter - 1) + 1\nquarter_start = pd.Timestamp(year=max_date.year, month=start_month, day=1)\nmask = (df['date'] >= quarter_start) & (df['date'] <= max_date)\nlast_q = df.loc[mask]\n# Get worst 5 products by total quantity\nworst5 = (last_q.groupby('product')['quantity']\n               .sum()\n               .nsmallest(5)\n               .reset_index())\nprint('Worst 5 products:')\nprint(worst5)\n# Filter last_q to those products\nfiltered = last_q[last_q['product'].isin(worst5['product'])]\n# Group by product and region, sum quantity\nbreakdown = (filtered.groupby(['product', 'region'])['quantity']\n                     .sum()\n                     .reset_index())\nprint('\\nBreakdown by region:')\nprint(breakdown)\n", "result": "Worst 5 products:\n               product

In [22]:
import sys; sys.path.insert(0, "src")
from agent.agent import ask
from agent.tools import FIGURES

answer, history = ask("Plot monthly revenue and tell me if there's a trend.", df)
print(answer)
FIGURES[-1]      # last line of the cell renders the interactive chart

{"question": "Plot monthly revenue and tell me if there's a trend.", "steps": [{"code": "import pandas as pd, plotly.express as px\n# Ensure df is defined\n# Create a month column\ndf['month'] = df['date'].dt.to_period('M').dt.to_timestamp()\nmonthly_rev = df.groupby('month')['revenue'].sum().reset_index()\nprint(monthly_rev)\nfig = px.line(monthly_rev, x='month', y='revenue', title='Monthly Revenue', labels={'month':'Month','revenue':'Revenue'})\nfig.update_xaxes(dtick='M1')\nfig.show()\n", "result": "       month   revenue\n0 2026-07-01  10078.09\n1 2026-08-01   5821.08\n2 2026-09-01  19881.96\n3 2026-10-01   6585.11\n4 2026-11-01   3903.02\n", "latency_s": 0.7, "error": false}], "answer": "**Monthly Revenue**\n\n| month      | revenue |\n|-----------|---------|\n| 2026\u201107\u201101 | 10078.09 |\n| 2026\u201108\u201101 | 5821.08 |\n| 2026\u201109\u201101 | 19881.96 |\n| 2026\u201110\u201101 | 6585.11 |\n| 2026\u201111\u201101 | 3903.02 |\n\nThe line chart (displayed above) shows t

**Monthly Revenue**

| month      | revenue |
|-----------|---------|
| 2026‑07‑01 | 10078.09 |
| 2026‑08‑01 | 5821.08 |
| 2026‑09‑01 | 19881.96 |
| 2026‑10‑01 | 6585.11 |
| 2026‑11‑01 | 3903.02 |

The line chart (displayed above) shows the revenue for each month in the dataset. The pattern is irregular: revenue spikes in September, then falls sharply in October and continues decreasing through November. There isn’t a steady upward or downward trend across the whole period; rather, revenue appears to fluctuate month‑to‑month.


IndexError: list index out of range